# Low-level single-point 2D functions

**Author:** Hannu Parviainen </br>
**Edited:** 12 June 2026

In [ ]:
from numpy import *
from matplotlib.pyplot import *

In [ ]:
sys.path.append('..')

In [ ]:
from matplotlib.pyplot import rc, subplots, setp
rc('figure', figsize=(13,4))

from meepmeep.numba2d import solve2d, solve2d_d, pos, pos_c, pos_d, pos_cd, sep, sep_d

In [ ]:
pars = 't0 p a i e w l'.split()

In [ ]:
t0, p, a, i, e, w, l = 0.20, 2.2, 8.0, 0.49*pi, 0.15, 0.25*pi, 0.0*pi
times = linspace(t0-0.1, t0+0.1, 1000)
tk = 0.0

## Solve the Taylor Series

First we solve the Taylor series at a given point. Let's also check that the value-only and derivative-returning `solve2d` coefficients agree with each other.

In [ ]:
c, dc = solve2d_d(tk, p, a, i, e, w, l)

round(c-solve2d(tk, p, a, i, e, w, l), 10)

### Sky-projected separation

In [ ]:
d, dd = sep_d(times, t0, p, c, dc, tk)

max(abs(d - sep(times, t0, p, c, tk)))

In [ ]:
fig, axs = subplots(2, 4, figsize=(13, 5), constrained_layout=True, sharex='all')
axs.flat[0].plot(times, d)

for j, dp in enumerate(dd.T):
    axs.flat[j+1].plot(times, dp)
    axs.flat[j+1].set_title(fr"$\delta$b / $\delta${pars[j]}")

### Sky-plane position

In [ ]:
x, y, dx, dy = pos_d(times, t0, p, c, dc, tk)

[max(abs(a1 - a2)) for a1, a2 in zip((x, y), pos(times, t0, p, c, tk))]

In [ ]:
fig, axs = subplots(4, 4, figsize=(13, 10), constrained_layout=True, sharex='all')
axs.flat[0].plot(times, x)
for j, dp in enumerate(dx.T):
    axs.flat[j+1].plot(times, dp)
    axs.flat[j+1].set_title(fr"$\delta$x / $\delta${pars[j]}")


axs.flat[8].plot(times, y)
for j, dp in enumerate(dy.T):
    axs.flat[j+9].plot(times, dp)
    axs.flat[j+9].set_title(fr"$\delta$y / $\delta${pars[j]}")

## Parallelisation

In [ ]:
from meepmeep.backends.numba.point2d import _sep_v, _sep_vp
from meepmeep.backends.numba.point2dd import _sep_d_v, _sep_d_vp

In [ ]:
times_a = linspace(t0-0.1, t0+0.1, 1000)
times_b = linspace(t0-0.1, t0+0.1, 10000)
times_c = linspace(t0-0.1, t0+0.1, 50000)
times_d = linspace(t0-0.1, t0+0.1, 100000)

In [ ]:
from timeit import repeat

In [ ]:
for case in "abcd":
    t = repeat(f"_sep_v(times_{case}, t0, p, c, tk)", repeat=6, number=100, globals=globals())
    print(round(mean(t), 5))

In [ ]:
for case in "abcd":
    t = repeat(f"_sep_vp(times_{case}, t0, p, c, tk)", repeat=6, number=100, globals=globals())
    print(round(mean(t), 5))

In [ ]:
for case in "abcd":
    t = repeat(f"_sep_d_v(times_{case}, t0, p, c, dc, tk)", repeat=6, number=100, globals=globals())
    print(round(mean(t), 5))

In [ ]:
for case in "abcd":
    t = repeat(f"_sep_d_vp(times_{case}, t0, p, c, dc, tk)", repeat=6, number=100, globals=globals())
    print(round(mean(t), 5))

In [ ]:
%%timeit -n 200 -r 100
_sep_v(times_a, t0, p, c, tk)

In [ ]:
%%timeit -n 100 -r 100
_sep_v(times_b, t0, p, c, tk)

In [ ]:
%%timeit -n 100 -r 100
_sep_v(times_c, t0, p, c, tk)

In [ ]:
%%timeit -n 100 -r 100
_sep_v(times_d, t0, p, c, tk)

In [ ]:
%%timeit -n 100 -r 100
_sep_vp(times_a, t0, p, c, tk)

In [ ]:
%%timeit -n 100 -r 100
_sep_vp(times_b, t0, p, c, tk)

In [ ]:
%%timeit -n 100 -r 100
_sep_vp(times_c, t0, p, c, tk)

In [ ]:
%%timeit -n 100 -r 100
_sep_vp(times_d, t0, p, c, tk)

---

<center> &copy;2026 Hannu Parviainen </center>